In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.stats import mannwhitneyu

# yoy: colunas esperadas -> generic_clean, group ('Shortage'|'Control'),
#                          spend_prev, spend_curr, yoy_pct
MIN_BASE = 10_000  # piso de gasto base; variações sobre baselines quase zero são ruído
df = yoy[yoy["spend_prev"] >= MIN_BASE].copy()
df["log_ratio"] = np.log10(df["spend_curr"] / df["spend_prev"])

s = df.loc[df.group == "Shortage", "yoy_pct"]
c = df.loc[df.group == "Control", "yoy_pct"]
_, p = mannwhitneyu(s, c, alternative="two-sided")

plt.style.use("dark_background")
fig = plt.figure(figsize=(16, 9), facecolor="#0f172a")
gs = GridSpec(2, 4, figure=fig, height_ratios=[1, 3], hspace=0.45, wspace=0.35)

# --- KPIs (medianas, não médias) ---
kpis = [
    ("Eventos Shortage (válidos)", f"{len(s)}"),
    ("Mediana YoY - Shortage", f"{s.median():+.1f}%"),
    ("Mediana YoY - Controle", f"{c.median():+.1f}%"),
    ("p-value (Mann-Whitney U)", f"{p:.3f}"),
]
for i, (lab, val) in enumerate(kpis):
    ax = fig.add_subplot(gs[0, i])
    ax.axis("off")
    ax.text(0.5, 0.62, val, ha="center", va="center", fontsize=26,
            weight="bold", color="white", transform=ax.transAxes)
    ax.text(0.5, 0.15, lab, ha="center", va="center", fontsize=12,
            color="#94a3b8", transform=ax.transAxes)

# --- Painel A: distribuição por grupo (escala log da razão) ---
ax1 = fig.add_subplot(gs[1, :2])
groups = ["Shortage", "Control"]
colors = {"Shortage": "#f87171", "Control": "#38bdf8"}
data = [df.loc[df.group == g, "log_ratio"].values for g in groups]
ax1.boxplot(data, widths=0.4, showfliers=False, patch_artist=True,
            boxprops=dict(facecolor="#1e293b", color="#94a3b8"),
            medianprops=dict(color="white", linewidth=2),
            whiskerprops=dict(color="#94a3b8"), capprops=dict(color="#94a3b8"))
rng = np.random.default_rng(42)
for i, (d, g) in enumerate(zip(data, groups), start=1):
    ax1.scatter(rng.normal(i, 0.04, len(d)), d, s=35, alpha=0.8, color=colors[g])
ax1.axhline(0, ls="--", color="#64748b")
ax1.set_xticks([1, 2], groups)
ax1.set_ylabel("log10(gasto t / gasto t-1)")
ax1.set_title(f"Variação YoY por grupo (n={len(s)} vs {len(c)}, baseline ≥ ${MIN_BASE:,})")

# --- Painel B: top 10 mais voláteis ---
ax2 = fig.add_subplot(gs[1, 2:])
top = (df.assign(absr=df.log_ratio.abs())
         .nlargest(10, "absr")
         .iloc[::-1])
bar_colors = [colors[g] for g in top.group]
ax2.barh(top.generic_clean, top.log_ratio, color=bar_colors)
for y, (v, pct) in enumerate(zip(top.log_ratio, top.yoy_pct)):
    ax2.text(v, y, f"  {pct:+,.0f}%", va="center", fontsize=9, color="white",
             ha="left" if v >= 0 else "right")
ax2.axvline(0, color="#64748b")
ax2.margins(x=0.2)
ax2.set_xlabel("log10(razão do gasto)")
ax2.set_title("Top 10 variações mais extremas")

fig.suptitle("FDA Drug Shortages vs. Medicare Part D Spending (2019-2023)",
             color="white", fontsize=20, weight="bold", y=0.98)
fig.savefig("../reports/figures/executive_dashboard_v2.png", dpi=200,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()

NameError: name 'yoy' is not defined